In [70]:
from __future__ import annotations

import argparse
import json
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import vedo
from scipy import ndimage
from skimage.filters import threshold_otsu
from vedo import settings

settings.default_backend = "vtk"

In [71]:
original_photos_path = Path("data\original_photos")
volumes = sorted(p for p in original_photos_path.rglob("*") if p.is_file() and p.suffix.lower() == ".tif")

# Segmentation

In [72]:
VIEWS = [
    ("+X", np.array([1, 0, 0]), np.array([0, 0, 1])),
    ("-X", np.array([-1, 0, 0]), np.array([0, 0, 1])),
    ("+Y", np.array([0, 1, 0]), np.array([0, 0, 1])),
    ("-Y", np.array([0, -1, 0]), np.array([0, 0, 1])),
    ("+Z", np.array([0, 0, 1]), np.array([0, 1, 0])),
    ("-Z", np.array([0, 0, -1]), np.array([0, 1, 0])),
]

def segment_largest_component(volume: np.ndarray) -> np.ndarray:
    # Otsu treshold for at fjerne vat
    threshold = threshold_otsu(volume)
    mask = volume > threshold

    # Dilation og fill holes for at beholde små dele
    mask = ndimage.binary_dilation(mask, iterations=5)
    mask = ndimage.binary_fill_holes(mask)
    
    # Finding connected compenents
    labeled, num = ndimage.label(mask)
    
    # Lægger værdier sammen for hvert component
    sizes = ndimage.sum_labels(volume, labeled, index=np.arange(1, num + 1))

    # Finder største vægtede komponent ikke largest component generelt.
    largest = int(np.argmax(sizes) + 1)
    mask = labeled == largest
    return mask

In [89]:
threshold_percentile = 97.5
zoom = 1.2
render_size = (980, 980)
view_angle_deg = 25.0

def render_views(clean_volume: np.ndarray, out_dir: Path, base_name: str) -> dict[str, object]:
    out_dir.mkdir(exist_ok=True)
    print(clean_volume.shape)
    vol = vedo.Volume(clean_volume, dims=clean_volume.shape)
    xmin, xmax, ymin, ymax, zmin, zmax = vol.bounds()

    # Finder de to yderste hjørner
    p_min = np.array([xmin, ymin, zmin])
    p_max = np.array([xmax, ymax, zmax])

    center = (p_min + p_max) / 2
    diag = np.linalg.norm(p_max - p_min)
    distance = diag * 1.5
    camera_views = []

    plotter = vedo.Plotter(size=render_size, offscreen=True, bg="white")
    try:
        for label, direction, view_up in VIEWS:
            plotter.clear()
            plotter.show(vol, resetcam=True, zoom=zoom)

            cam = plotter.camera
            cam.SetFocalPoint(*center)
            cam.SetPosition(*(center + direction * distance))
            cam.SetViewUp(*view_up)
            cam.SetViewAngle(view_angle_deg)
            plotter.renderer.ResetCameraClippingRange()

            out_path = out_dir / f"{base_name}_{label}.png"
            plotter.screenshot(str(out_path))
            cam_pos = (center + direction * distance).tolist()
            camera_views.append(
                {
                    "angle": label,
                    "direction": direction.astype(float).tolist(),
                    "view_up": view_up.astype(float).tolist(),
                    "camera_position": cam_pos,
                    "camera_focal_point": center.astype(float).tolist(),
                    "view_angle_deg": float(cam.GetViewAngle()),
                    "output_image": str(out_path),
                }
            )
    finally:
        plotter.close()

    return

volume = vedo.load(volumes[0]).tonumpy()
volume_name = volumes[0].parts[-1][:-4]

mask = segment_largest_component(volume)
volume = volume * mask
render_views(volume, Path(f"data/new_photos/segmented/AC/{volume_name}"), volume_name)

(256, 256, 512)


# Cosine similarity med DINOv3